# Schema-Aware NL2SQL — QLoRA fine-tune a code LLM on Spider (higher accuracy)

The **decoder-only** alternative to `finetune_colab.ipynb` (which fine-tunes T5). This fine-tunes **Qwen2.5-Coder** in **4-bit QLoRA** — far stronger at SQL than t5-base/large, and it still fits a **free Colab T4**. No Unsloth, no TRL — just transformers + peft + bitsandbytes.

**Before you start:**
1. `Runtime → Change runtime type → T4 GPU`.
2. Have a **Hugging Face write token** ready (add it as a Colab Secret named `HF_TOKEN`).
3. Download **`spider.zip`** once from the official Spider release; upload it in Step 3 (or set a Google Drive file id).

## Step 0 — Confirm the GPU

In [ ]:
!nvidia-smi

## Step 1 — Settings (edit these)

In [ ]:
REPO_BRANCH = "feature/schema-aware-anydb"
# 3B is the sweet spot for a FREE T4: strong on SQL (~70%) and a full run finishes in ~2-3h.
# The 7B is stronger (~80%) but ~10h on a T4 -> it will NOT finish a free 5h session. Use the 7B
# only on Colab Pro / an L4 or A100. Smaller/faster still: Qwen/Qwen2.5-Coder-1.5B-Instruct.
BASE_MODEL  = "Qwen/Qwen2.5-Coder-3B-Instruct"
EPOCHS      = 2
MODEL_REPO  = "Srijan-Ratrey/nl2sql-qwen-coder-spider"   # where the LoRA adapter gets pushed (Step 7)

# Optional: Google Drive file id for spider.zip. Leave "" to upload manually in Step 3.
SPIDER_GDRIVE_ID = ""
print("base:", BASE_MODEL, "| epochs:", EPOCHS, "| push to:", MODEL_REPO)

## Step 2 — Clone the repo & install dependencies

In [ ]:
%cd /content
!rm -rf Schema-Aware-Natural-Language-to-SQL-Agent
!git clone -b {REPO_BRANCH} https://github.com/Srijan-Ratrey/Schema-Aware-Natural-Language-to-SQL-Agent.git
%cd /content/Schema-Aware-Natural-Language-to-SQL-Agent
# QLoRA needs bitsandbytes (4-bit) + peft/accelerate. datasets/sqlglot for data + SQL parsing.
!pip -q install "transformers>=4.44" datasets peft accelerate bitsandbytes sqlglot sentencepiece
%env HF_DATASETS_TRUST_REMOTE_CODE=1

# Authenticate to the HF Hub from Colab Secrets (left panel: add a secret named HF_TOKEN with a
# WRITE token and enable "Notebook access").
import os
try:
    from google.colab import userdata
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
    from huggingface_hub import login
    login(token=os.environ["HF_TOKEN"])
    print("HF auth OK")
except Exception as e:
    print("HF auth skipped (set an HF_TOKEN secret to enable):", e)

## Step 3 — Get the Spider data (schemas + databases)

Needs `tables.json` (schemas) and `database/` (the SQLite DBs, for execution-accuracy eval).

In [ ]:
import os, glob, zipfile

if SPIDER_GDRIVE_ID:
    !pip -q install gdown
    !gdown {SPIDER_GDRIVE_ID} -O /content/spider.zip
else:
    from google.colab import files
    print("Upload spider.zip (from the official Spider release)...")
    files.upload()

for z in glob.glob("/content/spider*.zip") + glob.glob("spider*.zip"):
    print("Extracting", z)
    with zipfile.ZipFile(z) as f:
        f.extractall("/content/spider_data")

_tables = glob.glob("/content/**/tables.json", recursive=True)
TABLES_JSON = os.path.abspath(_tables[0]) if _tables else None
_db_dirs = [d for d in glob.glob("/content/**/database", recursive=True) if os.path.isdir(d)]
DB_DIR = os.path.abspath(_db_dirs[0]) if _db_dirs else None
print("TABLES_JSON:", TABLES_JSON)
print("DB_DIR:", DB_DIR)
assert TABLES_JSON and DB_DIR, "Could not find tables.json / database/ inside the zip."

## Step 4 — Smoke test (~2 min)
Validates train → save → eval on a tiny sample before committing real compute. **Expect ~0% accuracy here** — 200 samples / 1 epoch is far too little; this only confirms the pipeline runs without errors.

In [ ]:
!python scripts/train_qlora.py \
  --base-model "{BASE_MODEL}" \
  --tables-json "{TABLES_JSON}" \
  --epochs 1 --max-train-samples 200 \
  --output-dir smoke-qlora

## Step 5 — Full fine-tune
Saves the LoRA adapter to `nl2sql-qwen-qlora/` (checkpoint each epoch, best kept via early stopping). The 3B at `--batch-size 4 --max-len 768` uses ~9–11 GB on a T4 and finishes ~2 epochs in **~2–4 h** — inside a free session.

**Why these numbers:** Qwen's vocabulary is ~152k, so the loss step materializes a huge `batch × seq × vocab` logits tensor in fp32 — that's what OOMs, not the model weights. Batch 4 + a 768 length cap keeps it in budget. **Still OOM?** drop to `--batch-size 2 --grad-accum 8`. The stronger 7B needs Colab Pro / L4 / A100 (won't finish a free T4 session).

In [ ]:
!python scripts/train_qlora.py \
  --base-model "{BASE_MODEL}" \
  --tables-json "{TABLES_JSON}" \
  --epochs {EPOCHS} \
  --batch-size 4 --grad-accum 4 \
  --max-len 768 \
  --output-dir nl2sql-qwen-qlora

## Step 6 — Execution-accuracy evaluation
Same eval as the T5 path, with `--causal`. Runs predicted vs. gold SQL against the real DBs and writes the number + samples to `docs/EVAL.md`. Uses **greedy decoding** (`--num-beams 1`) — beam search on a 7B in 4-bit would OOM/crawl on a T4. Drop `--limit` for the full dev set (slower).

In [ ]:
!python scripts/evaluate_spider.py --causal \
  --base-model "{BASE_MODEL}" --adapter ./nl2sql-qwen-qlora \
  --tables-json "{TABLES_JSON}" \
  --spider-db-dir "{DB_DIR}" \
  --num-beams 1 \
  --limit 200

print("\n===== docs/EVAL.md =====")
print(open("docs/EVAL.md").read())

## Step 7 — Push the LoRA adapter to the Hugging Face Hub
Uploads the **adapter** you already trained (small, ~100–200 MB) — no retraining, no merge. This is the reliable option on a free Colab; merging a 7B into fp16 needs ~14 GB RAM the free tier doesn't have. At serving time you load the base + this adapter (or merge later on a bigger machine). Uses the `HF_TOKEN` secret from Step 2.

In [ ]:
# Push the already-trained adapter directly — no retraining, no merge.
from huggingface_hub import HfApi
api = HfApi()
api.create_repo(MODEL_REPO, exist_ok=True)
api.upload_folder(folder_path="nl2sql-qwen-qlora", repo_id=MODEL_REPO)
print("Pushed adapter to:", MODEL_REPO)
print("Serve with: base =", BASE_MODEL, "+ adapter =", MODEL_REPO)

## Done 🎉

Tip: download the adapter so you don't lose it when the runtime recycles:
```python
from google.colab import files; import shutil
shutil.make_archive('nl2sql-qwen-qlora', 'zip', 'nl2sql-qwen-qlora'); files.download('nl2sql-qwen-qlora.zip')
```